In [1]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# CELL 2: PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project"
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

# Create processed-data directory
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed data directory:")
print(PROCESSED_DIR)

Project root:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project

Processed data directory:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\data\processed


In [3]:
# ============================================================
# CELL 3: LOAD TRAINING DATA
# ============================================================

df = pd.read_csv(TRAIN_PATH)

# Clean column names only
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

print("Training dataset loaded.")
print()

print("Shape:", df.shape)

print("\nColumns:")
print(list(df.columns))

Training dataset loaded.

Shape: (404290, 6)

Columns:
['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']


In [4]:
# ============================================================
# CELL 4: VALIDATE REQUIRED COLUMNS
# ============================================================

required_columns = [
    "question1",
    "question2",
    "is_duplicate"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(missing_columns)
    )

print("✓ question1")
print("✓ question2")
print("✓ is_duplicate")

if "qid1" in df.columns:
    print("✓ qid1")

if "qid2" in df.columns:
    print("✓ qid2")

print("\nDataset validation successful.")

✓ question1
✓ question2
✓ is_duplicate
✓ qid1
✓ qid2

Dataset validation successful.


In [5]:
# ============================================================
# CELL 5: SELECT REQUIRED COLUMNS
# ============================================================

columns_to_keep = []

for column in [
    "id",
    "qid1",
    "qid2",
    "question1",
    "question2",
    "is_duplicate"
]:
    if column in df.columns:
        columns_to_keep.append(column)

df = df[columns_to_keep].copy()

print("Columns retained:")
print(columns_to_keep)

print("\nShape:")
print(df.shape)

Columns retained:
['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']

Shape:
(404290, 6)


In [6]:
# ============================================================
# CELL 6: STANDARDIZE TARGET
# ============================================================

print("Original target values:")
print(df["is_duplicate"].value_counts(dropna=False))

# Convert numeric-looking values to numeric
df["is_duplicate"] = pd.to_numeric(
    df["is_duplicate"],
    errors="coerce"
)

invalid_target = ~df["is_duplicate"].isin([0, 1])

print(
    "\nInvalid target rows:",
    invalid_target.sum()
)

Original target values:
is_duplicate
0    255027
1    149263
Name: count, dtype: int64

Invalid target rows: 0


In [7]:
# ============================================================
# CELL 7: REMOVE INVALID TARGET ROWS
# ============================================================

before = len(df)

df = df[
    df["is_duplicate"].isin([0, 1])
].copy()

after = len(df)

print("Rows before:", f"{before:,}")
print("Rows after :", f"{after:,}")
print("Removed    :", f"{before - after:,}")

Rows before: 404,290
Rows after : 404,290
Removed    : 0


In [8]:
# ============================================================
# CELL 8: REMOVE MISSING QUESTIONS
# ============================================================

before = len(df)

missing_questions = (
    df["question1"].isna()
    |
    df["question2"].isna()
)

print(
    "Rows with missing questions:",
    missing_questions.sum()
)

df = df[
    ~missing_questions
].copy()

print(
    "\nRows removed:",
    before - len(df)
)

print(
    "Remaining rows:",
    len(df)
)

Rows with missing questions: 3

Rows removed: 3
Remaining rows: 404287


In [9]:
# ============================================================
# CELL 9: CONVERT QUESTIONS TO STRING
# ============================================================

df["question1"] = (
    df["question1"]
    .astype(str)
)

df["question2"] = (
    df["question2"]
    .astype(str)
)

print("Question columns converted to string.")

Question columns converted to string.


In [10]:
# ============================================================
# CELL 10: TEXT NORMALIZATION FUNCTION
# ============================================================

def normalize_question(text):
    """
    Basic normalization only.

    This function:
    - converts text to lowercase
    - removes HTML tags
    - normalizes whitespace
    - removes leading/trailing spaces

    It does NOT perform:
    - stemming
    - lemmatization
    - TF-IDF
    - embeddings
    - similarity calculation
    """

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Strip spaces
    text = text.strip()

    return text


print("Normalization function created.")

Normalization function created.


In [11]:
# ============================================================
# CELL 11: CREATE CLEAN QUESTIONS
# ============================================================

df["question1_clean"] = (
    df["question1"]
    .apply(normalize_question)
)

df["question2_clean"] = (
    df["question2"]
    .apply(normalize_question)
)

print("Clean question columns created.")

display(
    df[
        [
            "question1",
            "question1_clean",
            "question2",
            "question2_clean"
        ]
    ].head(10)
)

Clean question columns created.


,question1,question1_clean,question2,question2_clean
0,What is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,what is the story of kohinoor (koh-i-noor) dia...,What would happen if the Indian government sto...,what would happen if the indian government sto...
2,How can I increase the speed of my internet co...,how can i increase the speed of my internet co...,How can Internet speed be increased by hacking...,how can internet speed be increased by hacking...
3,Why am I mentally very lonely? How can I solve...,why am i mentally very lonely? how can i solve...,Find the remainder when [math]23^{24}[/math] i...,find the remainder when [math]23^{24}[/math] i...
4,"Which one dissolve in water quikly sugar, salt...","which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,which fish would survive in salt water?
5,Astrology: I am a Capricorn Sun Cap moon and c...,astrology: i am a capricorn sun cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan...","i'm a triple capricorn (sun, moon and ascendan..."
6,Should I buy tiago?,should i buy tiago?,What keeps childern active and far from phone ...,what keeps childern active and far from phone ...
7,How can I be a good geologist?,how can i be a good geologist?,What should I do to be a great geologist?,what should i do to be a great geologist?
8,When do you use シ instead of し?,when do you use シ instead of し?,"When do you use ""&"" instead of ""and""?","when do you use ""&"" instead of ""and""?"
9,Motorola (company): Can I hack my Charter Moto...,motorola (company): can i hack my charter moto...,How do I hack Motorola DCX3400 for free internet?,how do i hack motorola dcx3400 for free internet?


In [12]:
# ============================================================
# CELL 12: CHECK EMPTY QUESTIONS
# ============================================================

empty_q1 = (
    df["question1_clean"]
    .eq("")
)

empty_q2 = (
    df["question2_clean"]
    .eq("")
)

print(
    "Empty Question 1:",
    empty_q1.sum()
)

print(
    "Empty Question 2:",
    empty_q2.sum()
)

Empty Question 1: 0
Empty Question 2: 0


In [13]:
# ============================================================
# CELL 13: REMOVE EMPTY QUESTIONS
# ============================================================

before = len(df)

df = df[
    (~empty_q1) &
    (~empty_q2)
].copy()

after = len(df)

print("Rows before:", f"{before:,}")
print("Rows after :", f"{after:,}")
print("Removed    :", f"{before - after:,}")

Rows before: 404,287
Rows after : 404,287
Removed    : 0


In [14]:
# ============================================================
# CELL 14: CREATE CANONICAL PAIR KEY
# ============================================================

df["_q_low"] = np.where(
    df["question1_clean"] <= df["question2_clean"],
    df["question1_clean"],
    df["question2_clean"]
)

df["_q_high"] = np.where(
    df["question1_clean"] <= df["question2_clean"],
    df["question2_clean"],
    df["question1_clean"]
)

df["pair_key"] = (
    df["_q_low"]
    + " ||| "
    + df["_q_high"]
)

print("Canonical pair keys created.")

Canonical pair keys created.


In [15]:
# ============================================================
# CELL 15: REPEATED PAIR ANALYSIS
# ============================================================

pair_counts = (
    df["pair_key"]
    .value_counts()
)

repeated_pairs = (
    pair_counts > 1
)

print(
    "Total unique pairs:",
    len(pair_counts)
)

print(
    "Pairs occurring more than once:",
    repeated_pairs.sum()
)

print(
    "Rows belonging to repeated pairs:",
    pair_counts[
        pair_counts > 1
    ].sum()
)

Total unique pairs: 403746
Pairs occurring more than once: 514
Rows belonging to repeated pairs: 1055


In [16]:
# ============================================================
# CELL 16: FIND CONFLICTING PAIR LABELS
# ============================================================

pair_label_counts = (
    df.groupby("pair_key")["is_duplicate"]
    .nunique()
)

conflicting_keys = (
    pair_label_counts[
        pair_label_counts > 1
    ]
    .index
)

print(
    "Conflicting pair groups:",
    len(conflicting_keys)
)

Conflicting pair groups: 10


In [17]:
# ============================================================
# CELL 17: REMOVE CONFLICTING PAIRS
# ============================================================

before = len(df)

if len(conflicting_keys) > 0:

    df = df[
        ~df["pair_key"].isin(
            conflicting_keys
        )
    ].copy()

after = len(df)

print("Rows before:", f"{before:,}")
print("Rows after :", f"{after:,}")
print("Removed    :", f"{before - after:,}")

Rows before: 404,287
Rows after : 404,267
Removed    : 20


In [18]:
# ============================================================
# CELL 18: REMOVE DUPLICATE PAIR ROWS
# ============================================================

before = len(df)

df = (
    df
    .drop_duplicates(
        subset=[
            "pair_key",
            "is_duplicate"
        ]
    )
    .copy()
)

after = len(df)

print("Rows before:", f"{before:,}")
print("Rows after :", f"{after:,}")
print("Duplicate rows removed:", f"{before - after:,}")

Rows before: 404,267
Rows after : 403,736
Duplicate rows removed: 531


In [19]:
# ============================================================
# CELL 19: CHECK TARGET AFTER CLEANING
# ============================================================

print(
    df["is_duplicate"]
    .value_counts()
)

print()

print(
    (
        df["is_duplicate"]
        .value_counts(
            normalize=True
        )
        .mul(100)
        .round(2)
    )
)

is_duplicate
0    254984
1    148752
Name: count, dtype: int64

is_duplicate
0    63.16
1    36.84
Name: proportion, dtype: float64


In [20]:
# ============================================================
# CELL 20: RESET INDEX
# ============================================================

df = df.reset_index(
    drop=True
)

print(
    "Index reset."
)

print(
    "Current rows:",
    f"{len(df):,}"
)

Index reset.
Current rows: 403,736


In [32]:
# ============================================================
# CELL 21: PREPARE QUESTION GROUPING
# ============================================================

print("Preparing leakage-safe question grouping...")

# Make sure cleaned questions are strings
df["question1_clean"] = (
    df["question1_clean"]
    .fillna("")
    .astype(str)
)

df["question2_clean"] = (
    df["question2_clean"]
    .fillna("")
    .astype(str)
)

print("✓ Clean question columns verified.")

Preparing leakage-safe question grouping...
✓ Clean question columns verified.


In [33]:
# ============================================================
# CELL 22: BUILD TEXT-BASED QUESTION COMPONENTS
# ============================================================

class UnionFind:

    def __init__(self):
        self.parent = {}
        self.rank = {}

    def add(self, x):

        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0

    def find(self, x):

        if self.parent[x] != x:
            self.parent[x] = self.find(
                self.parent[x]
            )

        return self.parent[x]

    def union(self, a, b):

        self.add(a)
        self.add(b)

        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        if self.rank[root_a] < self.rank[root_b]:

            self.parent[root_a] = root_b

        elif self.rank[root_a] > self.rank[root_b]:

            self.parent[root_b] = root_a

        else:

            self.parent[root_b] = root_a
            self.rank[root_a] += 1


# Create Union-Find structure
uf = UnionFind()

# Connect Question 1 with Question 2
for q1, q2 in zip(
    df["question1_clean"],
    df["question2_clean"]
):

    uf.union(q1, q2)


# Assign component ID to every pair
df["question_group"] = [
    uf.find(q1)
    for q1 in df["question1_clean"]
]


print(
    "✓ Text-based question groups created."
)

print(
    "Unique question groups:",
    df["question_group"].nunique()
)

✓ Text-based question groups created.
Unique question groups: 200336


In [34]:
# ============================================================
# CELL 23: QUESTION GROUP SIZE ANALYSIS
# ============================================================

group_sizes = (
    df["question_group"]
    .value_counts()
)

print(
    "Number of question groups:",
    len(group_sizes)
)

print("\nGroup size statistics:")

display(
    group_sizes.describe()
)

print("\nLargest question groups:")

display(
    group_sizes.head(10)
)

Number of question groups: 200336

Group size statistics:


count    200336.000000
mean          2.015294
std          42.396561
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max       17527.000000
Name: count, dtype: float64


Largest question groups:


question_group
how can i find the number of instagram accounts with more than 10000 followers?                  17527
how did donald trump manage to win the 2016 us election?                                          3813
why did indian government scrap rs 1000 and 500 note and instead is introducing rs 2000 note?     3045
what are the best ways to improve my writing skills in english?                                   2495
what are some of the best horror movies?                                                          2033
what is easy way to earn money without any investments?                                           1727
is there a way to learn programming language quickly? how can i do it?                            1457
what is the most important thing in our life?                                                     1067
why do people ask questions on quora that are easily to find answers too on google?               1045
how do i post a question that was marked as needing improv

In [35]:
# ============================================================
# CELL 24: GROUP-AWARE SPLITTING
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

print(
    "GroupShuffleSplit imported successfully."
)

GroupShuffleSplit imported successfully.


In [36]:
# ============================================================
# CELL 25: TRAIN / TEMPORARY SPLIT
# ============================================================

gss_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=RANDOM_STATE
)

train_indices, temp_indices = next(
    gss_1.split(
        df,
        y=df["is_duplicate"],
        groups=df["question_group"]
    )
)

train_df = (
    df.iloc[train_indices]
    .copy()
)

temp_df = (
    df.iloc[temp_indices]
    .copy()
)

print(
    "Training rows:",
    f"{len(train_df):,}"
)

print(
    "Temporary rows:",
    f"{len(temp_df):,}"
)

Training rows: 271,524
Temporary rows: 132,212


In [37]:
# ============================================================
# CELL 26: VALIDATION / INTERNAL TEST SPLIT
# ============================================================

gss_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=RANDOM_STATE
)

val_indices, internal_test_indices = next(
    gss_2.split(
        temp_df,
        y=temp_df["is_duplicate"],
        groups=temp_df["question_group"]
    )
)

val_df = (
    temp_df.iloc[val_indices]
    .copy()
)

internal_test_df = (
    temp_df.iloc[internal_test_indices]
    .copy()
)

print(
    "Training rows:",
    f"{len(train_df):,}"
)

print(
    "Validation rows:",
    f"{len(val_df):,}"
)

print(
    "Internal test rows:",
    f"{len(internal_test_df):,}"
)

Training rows: 271,524
Validation rows: 76,226
Internal test rows: 55,986


In [38]:
# ============================================================
# CELL 27: SPLIT PERCENTAGES
# ============================================================

total = (
    len(train_df)
    + len(val_df)
    + len(internal_test_df)
)

split_summary = pd.DataFrame({
    "split": [
        "train",
        "validation",
        "internal_test"
    ],
    "rows": [
        len(train_df),
        len(val_df),
        len(internal_test_df)
    ]
})

split_summary["percentage"] = (
    split_summary["rows"]
    / total
    * 100
).round(2)

display(
    split_summary
)

,split,rows,percentage
0,train,271524,67.25
1,validation,76226,18.88
2,internal_test,55986,13.87


In [39]:
# ============================================================
# CELL 28: TARGET DISTRIBUTION BY SPLIT
# ============================================================

def target_summary(name, data):

    return {
        "split": name,
        "rows": len(data),
        "different_0_%": round(
            (data["is_duplicate"] == 0).mean() * 100,
            2
        ),
        "duplicate_1_%": round(
            (data["is_duplicate"] == 1).mean() * 100,
            2
        )
    }


split_target = pd.DataFrame([
    target_summary(
        "train",
        train_df
    ),
    target_summary(
        "validation",
        val_df
    ),
    target_summary(
        "internal_test",
        internal_test_df
    )
])

display(
    split_target
)

,split,rows,different_0_%,duplicate_1_%
0,train,271524,63.51,36.49
1,validation,76226,60.13,39.87
2,internal_test,55986,65.55,34.45


In [41]:
# ============================================================
# CELL 30: FINAL LEAKAGE VALIDATION
# ============================================================

if len(train_validation_overlap) > 0:

    print(
        "WARNING: Train/validation question overlap detected."
    )

elif len(train_test_overlap) > 0:

    print(
        "WARNING: Train/internal-test question overlap detected."
    )

elif len(validation_test_overlap) > 0:

    print(
        "WARNING: Validation/internal-test question overlap detected."
    )

else:

    print(
        "✓ NO QUESTION LEAKAGE DETECTED"
    )

    print(
        "✓ Training, validation and internal test "
        "contain separate normalized questions."
    )

✓ NO QUESTION LEAKAGE DETECTED
✓ Training, validation and internal test contain separate normalized questions.


In [42]:
# ============================================================
# CELL 30A: SHOW OVERLAPPING QUESTIONS
# ============================================================

if len(train_validation_overlap) > 0:

    print("Examples of Train ↔ Validation overlap:\n")

    for question in list(
        train_validation_overlap
    )[:20]:

        print(
            "-",
            question
        )

elif len(train_test_overlap) > 0:

    print("Examples of Train ↔ Internal Test overlap:\n")

    for question in list(
        train_test_overlap
    )[:20]:

        print(
            "-",
            question
        )

elif len(validation_test_overlap) > 0:

    print("Examples of Validation ↔ Internal Test overlap:\n")

    for question in list(
        validation_test_overlap
    )[:20]:

        print(
            "-",
            question
        )

else:

    print(
        "No overlapping questions."
    )

No overlapping questions.


In [43]:
# ============================================================
# CELL 31: REMOVE INTERNAL HELPER COLUMNS
# ============================================================

helper_columns = [
    "_q_low",
    "_q_high",
    "pair_key",
    "question_group"
]

for column in helper_columns:

    for dataset in [
        train_df,
        val_df,
        internal_test_df
    ]:

        if column in dataset.columns:
            dataset.drop(
                columns=[column],
                inplace=True
            )

print(
    "✓ Internal preprocessing columns removed."
)

✓ Internal preprocessing columns removed.


In [44]:
# ============================================================
# CELL 32: FINAL COLUMN ORDER
# ============================================================

final_columns = []

for column in [
    "id",
    "qid1",
    "qid2",
    "question1",
    "question2",
    "question1_clean",
    "question2_clean",
    "is_duplicate"
]:

    if column in train_df.columns:
        final_columns.append(column)

train_df = train_df[
    final_columns
].copy()

val_df = val_df[
    final_columns
].copy()

internal_test_df = internal_test_df[
    final_columns
].copy()

print(
    "Final columns:"
)

print(final_columns)

Final columns:
['id', 'qid1', 'qid2', 'question1', 'question2', 'question1_clean', 'question2_clean', 'is_duplicate']


In [45]:
# ============================================================
# CELL 33: RESET SPLIT INDEXES
# ============================================================

train_df = train_df.reset_index(
    drop=True
)

val_df = val_df.reset_index(
    drop=True
)

internal_test_df = (
    internal_test_df
    .reset_index(drop=True)
)

print("Indexes reset successfully.")

Indexes reset successfully.


In [46]:
# ============================================================
# CELL 34: FINAL QUALITY CHECK
# ============================================================

datasets = {
    "TRAIN": train_df,
    "VALIDATION": val_df,
    "INTERNAL TEST": internal_test_df
}

for name, data in datasets.items():

    print("=" * 70)
    print(name)
    print("=" * 70)

    print(
        "Rows:",
        f"{len(data):,}"
    )

    print(
        "Missing values:",
        data.isnull().sum().sum()
    )

    print(
        "Empty question1:",
        data["question1_clean"].eq("").sum()
    )

    print(
        "Empty question2:",
        data["question2_clean"].eq("").sum()
    )

    print(
        "Duplicate %:",
        round(
            data["is_duplicate"].mean() * 100,
            2
        )
    )

    print()

TRAIN
Rows: 271,524
Missing values: 0
Empty question1: 0
Empty question2: 0
Duplicate %: 36.49

VALIDATION
Rows: 76,226
Missing values: 0
Empty question1: 0
Empty question2: 0
Duplicate %: 39.87

INTERNAL TEST
Rows: 55,986
Missing values: 0
Empty question1: 0
Empty question2: 0
Duplicate %: 34.45



In [47]:
# ============================================================
# CELL 35: SAVE PROCESSED DATASETS
# ============================================================

TRAIN_PROCESSED_PATH = (
    PROCESSED_DIR / "train_processed.csv"
)

VAL_PROCESSED_PATH = (
    PROCESSED_DIR / "validation_processed.csv"
)

TEST_PROCESSED_PATH = (
    PROCESSED_DIR / "internal_test_processed.csv"
)

train_df.to_csv(
    TRAIN_PROCESSED_PATH,
    index=False
)

val_df.to_csv(
    VAL_PROCESSED_PATH,
    index=False
)

internal_test_df.to_csv(
    TEST_PROCESSED_PATH,
    index=False
)

print("Processed datasets saved:")
print()
print(TRAIN_PROCESSED_PATH)
print(VAL_PROCESSED_PATH)
print(TEST_PROCESSED_PATH)

Processed datasets saved:

C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\data\processed\train_processed.csv
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\data\processed\validation_processed.csv
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\data\processed\internal_test_processed.csv


In [48]:
# ============================================================
# CELL 36: SAVE PREPROCESSING CONFIGURATION
# ============================================================

preprocessing_config = {
    "random_state": RANDOM_STATE,

    "train_ratio": round(
        len(train_df) / total,
        4
    ),

    "validation_ratio": round(
        len(val_df) / total,
        4
    ),

    "internal_test_ratio": round(
        len(internal_test_df) / total,
        4
    ),

    "text_normalization": {
        "lowercase": True,
        "remove_html": True,
        "normalize_whitespace": True,
        "strip_spaces": True
    },

    "group_based_split": True,

    "target_column": "is_duplicate",

    "question_columns": [
        "question1_clean",
        "question2_clean"
    ]
}

CONFIG_PATH = (
    PROCESSED_DIR /
    "preprocessing_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        preprocessing_config,
        f,
        indent=4
    )

print(
    "Configuration saved to:"
)

print(CONFIG_PATH)

Configuration saved to:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\data\processed\preprocessing_config.json


In [49]:
# ============================================================
# CELL 37: VERIFY SAVED DATA
# ============================================================

train_check = pd.read_csv(
    TRAIN_PROCESSED_PATH
)

val_check = pd.read_csv(
    VAL_PROCESSED_PATH
)

test_check = pd.read_csv(
    TEST_PROCESSED_PATH
)

print(
    "Saved training shape:",
    train_check.shape
)

print(
    "Saved validation shape:",
    val_check.shape
)

print(
    "Saved internal test shape:",
    test_check.shape
)

Saved training shape: (271524, 8)
Saved validation shape: (76226, 8)
Saved internal test shape: (55986, 8)


In [50]:
# ============================================================
# CELL 38: FINAL PREPROCESSING SUMMARY
# ============================================================

print("=" * 75)
print("QUESTION SIMILARITY - PREPROCESSING COMPLETE")
print("=" * 75)

print()

print(
    f"Training samples      : {len(train_df):,}"
)

print(
    f"Validation samples    : {len(val_df):,}"
)

print(
    f"Internal test samples : {len(internal_test_df):,}"
)

print()

print(
    "Question leakage      : CHECKED"
)

print(
    "Text normalization    : COMPLETED"
)

print(
    "Invalid rows          : REMOVED"
)

print(
    "Conflicting pairs     : REMOVED"
)

print(
    "Duplicate pair rows   : REMOVED"
)

print(
    "Group-based split     : COMPLETED"
)

print()

print(
    "Processed files saved to:"
)

print(
    PROCESSED_DIR
)

QUESTION SIMILARITY - PREPROCESSING COMPLETE

Training samples      : 271,524
Validation samples    : 76,226
Internal test samples : 55,986

Question leakage      : CHECKED
Text normalization    : COMPLETED
Invalid rows          : REMOVED
Conflicting pairs     : REMOVED
Duplicate pair rows   : REMOVED
Group-based split     : COMPLETED

Processed files saved to:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\data\processed
